In [1]:
import pandas as pd, requests, zipfile, os

# ---------- Paramètres ----------
URL_MEN = "https://www.insee.fr/fr/statistiques/fichier/8268838/base-ic-logement-2021_csv.zip"
BASE = "/lakehouse/default/Files/menages"
BRUT = f"{BASE}/brut"
os.makedirs(BRUT, exist_ok=True)

# ---------- 1. Télécharger le recensement INSEE Logement 2021 ----------
archive = f"{BRUT}/base-ic-logement-2021_csv.zip"
if not os.path.exists(archive):
    r = requests.get(URL_MEN, timeout=600)
    r.raise_for_status()
    with open(archive, "wb") as f:
        f.write(r.content)
    print("Archive téléchargée")

# ---------- 2. Codes des communes des 2 villes (mêmes codes que 1_bronze_cartes) ----------
CODES_TOULON = [c["code"] for c in requests.get("https://geo.api.gouv.fr/epcis/248300543/communes", timeout=60).json()]
CODES_MARSEILLE = [str(c) for c in range(13201, 13217)]   # 16 arrondissements

# ---------- 3. Lire et garder Toulon + Marseille ----------
with zipfile.ZipFile(archive) as z:
    nom = [n for n in z.namelist() if n.lower().endswith(".csv") and not n.lower().startswith("meta")][0]
    df = pd.read_csv(z.open(nom), sep=";", dtype={"IRIS": str, "COM": str}, low_memory=False)

df["ville"] = None
df.loc[df.COM.isin(CODES_TOULON), "ville"] = "toulon"
df.loc[df.COM.isin(CODES_MARSEILLE), "ville"] = "marseille"
df = df[df.ville.notna()][["IRIS", "COM", "ville", "P21_MEN", "P21_RP", "P21_RP_VOIT1P", "P21_RP_VOIT1", "P21_RP_VOIT2P"]]

# ---------- 4. Indicateurs de motorisation (critère 1) ----------
df["menages_sans_voiture"] = df.P21_RP - df.P21_RP_VOIT1P
df["taux_sans_voiture"] = df.menages_sans_voiture / df.P21_RP
df["taux_0_ou_1_voiture"] = (df.menages_sans_voiture + df.P21_RP_VOIT1) / df.P21_RP

# ---------- 5. Résumé + sauvegarde ----------
print(df.groupby("ville").agg(
    iris=("IRIS", "count"),
    menages=("P21_RP", "sum"),
    sans_voiture=("menages_sans_voiture", "sum"),
))
df.to_parquet(f"{BASE}/menages_toulon_marseille.parquet")
print("Sauvegardé")

StatementMeta(, 501d451f-e87e-475f-9720-2fbf09ab59fd, 3, Finished, Available, Finished, False)

           iris        menages   sans_voiture
ville                                        
marseille   393  410236.597463  136127.972701
toulon      176  219615.111112   43017.503877
Sauvegardé
